In [ ]:
import numpy as np
import pandas as pd
import kagglehub
import os
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments
import torch
print(torch.__version__)

2.11.0+cpu


In [3]:
DATASET_DIR = './data'

In [4]:
if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)
    
path = kagglehub.dataset_download("alaakhaled/conll003-englishversion", output_dir=DATASET_DIR)
print(path)
print(os.listdir(path))

./data
['.complete', 'metadata', 'test.txt', 'train.txt', 'valid.txt']


In [5]:
def load_conll_ner(file_path):
    sentences = []
    tokens, labels = [], []
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            if not line:
                if tokens:
                    sentences.append({"tokens": tokens, "ner_tags": labels})
                    tokens, labels = [], []
                continue
            
            if line.startswith("-DOCSTART-"):
                continue
            
            
            word, pos, chunk, ner = line.split()
            tokens.append(word)
            labels.append(ner)

    # flush last sentence if file doesn't end with a blank line
    if tokens:
        sentences.append({"tokens": tokens, "ner_tags": labels})

    return sentences


train_data = load_conll_ner(f"{path}/train.txt")
val_data = load_conll_ner(f"{path}/valid.txt")
test_data  = load_conll_ner(f"{path}/test.txt")

train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)
test_dataset  = Dataset.from_list(test_data)

print(f"Number of training sentences: {len(train_dataset)}")
print(f"Number of validation sentences: {len(val_dataset)}")
print(f"Number of test sentences: {len(test_dataset)}")

print("Example sentence:")
print(train_dataset[0])

Number of training sentences: 14041
Number of validation sentences: 3250
Number of test sentences: 3453
Example sentence:
{'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'ner_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']}


In [6]:
# create label mapping

label_list = sorted(set(label for ex in train_dataset for label in ex["ner_tags"]))
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}
print("Label to ID mapping:")
print(label2id)
print("ID to Label mapping:")
print(id2label)


Label to ID mapping:
{'B-LOC': 0, 'B-MISC': 1, 'B-ORG': 2, 'B-PER': 3, 'I-LOC': 4, 'I-MISC': 5, 'I-ORG': 6, 'I-PER': 7, 'O': 8}
ID to Label mapping:
{0: 'B-LOC', 1: 'B-MISC', 2: 'B-ORG', 3: 'B-PER', 4: 'I-LOC', 5: 'I-MISC', 6: 'I-ORG', 7: 'I-PER', 8: 'O'}


In [7]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    word_ids = tokenized.word_ids()
    labels = []
    previous_word = None

    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)  # ignore
        elif word_id != previous_word:
            labels.append(label2id[example["ner_tags"][word_id]])
        else:
            labels.append(-100)  # ignore subword tokens

        previous_word = word_id

    tokenized["labels"] = labels
    return tokenized

train_dataset = train_dataset.map(tokenize_and_align_labels, batched=False)
train_dataset = train_dataset.remove_columns(["tokens", "ner_tags"])

val_dataset = val_dataset.map(tokenize_and_align_labels, batched=False)
val_dataset = val_dataset.remove_columns(["tokens", "ner_tags"])

print("Tokenized example:")
print(train_dataset[0])

Map: 100%|██████████| 3250/3250 [00:00<00:00, 4648.63 examples/s]

Tokenized example:
{'input_ids': [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 2, 8, 1, 8, 8, 8, 1, 8, -100, 8, -100]}


In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

batch = data_collator([
    train_dataset[0],
    train_dataset[1]
])
print(batch)

{'input_ids': tensor([[  101,  7270, 22961,  1528,  1840,  1106, 21423,  1418,  2495, 12913,
           119,   102],
        [  101,  1943, 14428,   102,     0,     0,     0,     0,     0,     0,
             0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]]), 'labels': tensor([[-100,    2,    8,    1,    8,    8,    8,    1,    8, -100,    8, -100],
        [-100,    3,    7, -100, -100, -100, -100, -100, -100, -100, -100, -100]])}


In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./ner_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7977.89it/s]
[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 